# Train the MLBB tactical network (free, in Google Colab)

Makes `policy.tflite` + `policy_info.json` for the MLBB Research app (put both in **Download/MLBB-Research**; the app loads them by itself).

It learns **from your own play**: the situation the app saw (HP, enemies, minions, monsters, tower circle, level, minimap...) and what you did (which of the 8 joystick directions or standing still, attack, skills, spell, recall). On the phone it runs in about a millisecond, 5-10 times a second, and its suggestions are mixed into the AI's rules (the safety rules - retreat, tower circle, recall - always come first).

**Before you start**
1. Play a few matches with the app's screen capture on (Coach mode records you whenever you play; more matches = better). A few hours per hero is a good start.
2. App → **Learn** page → *Export my recorded play (CSV)*. The file lands in Downloads/MLBB-Research.
3. *Runtime → Change runtime type*: CPU is enough (the network is tiny).


In [ ]:
# TensorFlow and pandas are already installed in Colab
import numpy as np, pandas as pd, tensorflow as tf, math, json, io
print("TensorFlow", tf.__version__)

## 1. Settings

In [ ]:
HERO = ""            # e.g. "Griffith" to train for one hero only; "" = all your heroes
WATCHED_WEIGHT = 0.2 # moments recorded from heroes you watched (allies while dead, enemies) count this much (0 = leave them out)
EPOCHS = 40
MIN_ROWS = 1500      # the richer 32-number situation is used when at least this many rows have it

## 2. Load your recorded play
Upload one or more `mlbb-my-play-*.csv` files (on a phone, the Google Drive cell below is more reliable).

In [ ]:
# A. Upload from this device
from google.colab import files
up = files.upload()
frames = [pd.read_csv(io.BytesIO(v)) for k, v in up.items() if k.endswith(".csv")]
df = pd.concat(frames, ignore_index=True)
print(len(df), "rows")

In [ ]:
# B. Or read them from Google Drive (put the CSV files in My Drive/MLBB-Research first)
# from google.colab import drive; import glob
# drive.mount("/content/drive")
# df = pd.concat([pd.read_csv(f) for f in glob.glob("/content/drive/MyDrive/MLBB-Research/mlbb-my-play-*.csv")], ignore_index=True)
# print(len(df), "rows")

## 3. Situation → action examples

In [ ]:
def parse(col):
    out = []
    for v in df[col].fillna("").astype(str):
        out.append([float(x) for x in v.split(",")] if v.strip() else None)
    return out

f1 = parse("features")
f2 = parse("features2") if "features2" in df.columns else [None] * len(df)
n40 = sum(1 for v in f2 if v is not None and len(v) == 40)
n32 = sum(1 for v in f2 if v is not None and len(v) in (32, 40))
# 40 numbers (v32+: also where you are going and which way you were moving) > 32 > 17
N = 40 if n40 >= MIN_ROWS else 32 if n32 >= MIN_ROWS else 17
use2 = N > 17
print(f"rows with 40 numbers: {n40}, with 32+: {n32} -> training on {N} inputs")

X, MOVE, CASTS, W, TS, HEROES = [], [], [], [], [], []
for i, row in df.iterrows():
    hero = str(row["hero"])
    if HERO and hero != HERO and not hero.startswith("~"):
        continue
    feat = f2[i] if use2 else f1[i]
    if feat is None or len(feat) < N:
        continue
    feat = feat[:N]
    mx, my = float(row["move_x"]), float(row["move_y"])
    # 8 joystick directions (0 = right, 2 = down, 4 = left, 6 = up) or 8 = stand still
    move = 8 if math.hypot(mx, my) < 0.4 else int(round(math.atan2(my, mx) / (math.pi / 4))) % 8
    c = int(row["casts"])
    X.append(feat); MOVE.append(move); CASTS.append([(c >> b) & 1 for b in range(6)])
    W.append(WATCHED_WEIGHT if hero.startswith("~") else 1.0)
    TS.append(int(row["ts"])); HEROES.append(hero)

X = np.array(X, np.float32); MOVE = np.array(MOVE); CASTS = np.array(CASTS, np.float32); W = np.array(W, np.float32)
TS = np.array(TS); HEROES = np.array(HEROES)
keep = W > 0
X, MOVE, CASTS, W, TS, HEROES = X[keep], MOVE[keep], CASTS[keep], W[keep], TS[keep], HEROES[keep]
# a press is recorded in one 0.2 s moment; the moments right before / after look the same. A press
# counts for its neighbours too (within 0.45 s, same recording), so "a little early / late" is not wrong.
order = np.argsort(TS, kind="stable"); X, MOVE, CASTS, W, TS, HEROES = X[order], MOVE[order], CASTS[order], W[order], TS[order], HEROES[order]
wide = CASTS.copy()
for d in (-1, 1):
    j = np.arange(len(TS)) + d
    ok = (j >= 0) & (j < len(TS))
    jj = np.clip(j, 0, len(TS) - 1)
    near = ok & (np.abs(TS[jj] - TS) <= 450) & (HEROES[jj] == HEROES)
    wide[near] = np.maximum(wide[near], CASTS[jj][near])
CASTS = wide
# the network's joystick is used in fights and around minions / monsters (walking routes stay with
# the planner), so it learns and is checked on those moments only
FIGHT = (X[:, 6] > 0.5) | (X[:, 7] > 0)
if N >= 32: FIGHT |= X[:, 20] > 0
print(f"moments with enemies / minions / monsters near: {FIGHT.sum()} of {len(X)}")
MOVE_NAMES = ["right", "down-right", "down", "down-left", "left", "up-left", "up", "up-right", "stand"]
BTN = ["attack", "skill 1", "skill 2", "ultimate", "spell", "recall"]
print(len(X), "examples")
cnt = np.bincount(MOVE, minlength=9)
print("joystick:", ", ".join(f"{n} {c} ({c / len(MOVE):.0%})" for n, c in zip(MOVE_NAMES, cnt)))
print("buttons: ", ", ".join(f"{n} {int(c)}" for n, c in zip(BTN, CASTS.sum(0))))
if cnt.max() / len(MOVE) > 0.7:
    print(f"NOTE: {cnt.max() / len(MOVE):.0%} of the moments are '{MOVE_NAMES[cnt.argmax()]}'. The training below balances this,"
          " but recording more varied play (walking, chasing, retreating) helps most.")
for n, c in zip(BTN, CASTS.sum(0)):
    if 0 < c < 50: print(f"NOTE: only {int(c)} '{n}' presses recorded - too few to learn it well yet")
assert len(X) >= 300, "Too little recorded play: play more matches with screen capture on, then export again."

# last 15% (the most recent play) checks the result
cut = int(len(X) * 0.85)
Xtr, Xva = X[:cut], X[cut:]

## 4. Train

In [ ]:
inp = tf.keras.Input(shape=(N,), name="situation")
h = tf.keras.layers.Dense(128, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.1)(h)
h = tf.keras.layers.Dense(64, activation="relu")(h)
move_out = tf.keras.layers.Dense(9, activation="softmax", name="move")(h)
cast_out = tf.keras.layers.Dense(6, activation="sigmoid", name="casts")(h)
model = tf.keras.Model(inp, [move_out, cast_out])

# Lopsided recordings (mostly one move, few skill presses) would teach it to always say the common
# move and never press skills. So every move counts as much as the others (rare ones weighted up,
# at most 15x) and a press counts as much as all the no-presses of that button (at most 30x).
F = FIGHT[:cut]
counts = np.bincount(MOVE[:cut][F], minlength=9)
present = counts > 0
cw = np.where(present, F.sum() / (present.sum() * np.maximum(counts, 1)), 0.0)
cw = np.clip(cw, 0.0, 15.0)
sw_move = W[:cut] * cw[MOVE[:cut]] * F
pos = CASTS[:cut].sum(0)
POS_W = tf.constant(np.clip((len(CASTS[:cut]) - pos) / np.maximum(pos, 1), 1.0, 30.0), tf.float32)
def weighted_bce(y, p):
    p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
    return -tf.reduce_mean(POS_W * y * tf.math.log(p) + (1 - y) * tf.math.log(1 - p), axis=-1)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=["sparse_categorical_crossentropy", weighted_bce],
              metrics=[["accuracy"], ["binary_accuracy"]])
hist = model.fit(Xtr, [MOVE[:cut], CASTS[:cut]],
                 sample_weight=[sw_move, W[:cut]],
                 validation_data=(Xva, [MOVE[cut:], CASTS[cut:]]),
                 epochs=EPOCHS, batch_size=128, verbose=2,
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

## 5. How good is it? (on your most recent play, which it did not train on)

In [ ]:
pm_all, pc = model.predict(Xva, verbose=0)
FV = FIGHT[cut:]
pm = pm_all[FV]; yv = MOVE[cut:][FV]
print(f"(joystick checked on the {FV.sum()} recent moments with enemies / minions / monsters near)")
acc = (pm.argmax(1) == yv).mean()
base = (yv == np.bincount(MOVE[:cut][FIGHT[:cut]], minlength=9).argmax()).mean()
# per-move: of the times you did it, how often it says it too
per = [((pm.argmax(1) == k) & (yv == k)).sum() / (yv == k).sum() for k in range(9) if (yv == k).sum() > 0]
bal = float(np.mean(per)); chance = 1 / len(per)
print(f"joystick: same as you {acc:.0%} of the time (always the common move: {base:.0%})")
print(f"joystick, every move counted equally: {bal:.0%}  (blind guessing: {chance:.0%})  <- the number that matters")
for k in range(9):
    n = (yv == k).sum()
    if n: print(f"   {MOVE_NAMES[k]:10s} {n:5d} times, it said the same {((pm.argmax(1) == k) & (yv == k)).sum() / n:.0%}")

# for each button, the press line that works best on your play (it is built into the exported file)
APP = np.array([0.5, 0.6, 0.6, 0.6, 0.6, 0.85], np.float32)    # the app's own press lines
THR = []
QUALITY = {}    # per button: how many presses it was checked on, how often its presses were right
for b, n in enumerate(BTN):
    y = CASTS[cut:, b] > 0
    if y.sum() < 5:
        # too few presses to tune: keep the app's own line (a never-pressed button stays unpressed)
        THR.append(float(APP[b]))
        print(f"{n:9s} " + ("never pressed in your recent play - cannot check" if y.sum() == 0 else f"only {int(y.sum())} presses in your recent play - not tuned"))
        continue
    best_t, best_f = float(APP[b]), -1.0
    for t in np.arange(0.05, 0.96, 0.05):
        p = pc[:, b] > t
        tp = (p & y).sum(); f = 2 * tp / max(1, p.sum() + y.sum())
        if f > best_f: best_f, best_t = f, float(t)
    THR.append(best_t)
    p = pc[:, b] > best_t
    QUALITY[["attack", "s1", "s2", "ult", "spell", "recall"][b]] = {"checked": int(y.sum()), "right": float((p & y).sum() / max(1, p.sum())), "found": float((p & y).sum() / y.sum())}
    print(f"{n:9s} when you pressed it (within 0.2 s): found {(p & y).sum() / y.sum():.0%}, right {(p & y).sum() / max(1, p.sum()):.0%} of its presses (press line {best_t:.2f})")

## 6. Export to LiteRT (.tflite) for the phone
One output of 15 numbers: 9 joystick choices (they add up to 1) then the 6 press chances.

In [ ]:
# The app presses a button when the network says at least 0.5 (attack), 0.6 (skills, spell) or 0.85
# (recall). The best press lines found above are built into the file so they match the app's.
logit = lambda v: np.log(v / (1 - v))
SHIFT = tf.constant(logit(APP) - logit(np.clip(np.array(THR, np.float32), 0.05, 0.95)), tf.float32)
class Calibrate(tf.keras.layers.Layer):
    def call(self, p):
        p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
        return tf.sigmoid(tf.math.log(p / (1 - p)) + SHIFT)
export = tf.keras.Model(inp, tf.keras.layers.Concatenate(name="policy")([move_out, Calibrate()(cast_out)]))
conv = tf.lite.TFLiteConverter.from_keras_model(export)
tfl = conv.convert()
open("/content/policy.tflite", "wb").write(tfl)
# how good it is, for the app: it only uses the joystick / the buttons this file says are reliable
moves_seen = np.bincount(yv, minlength=9)
json.dump({"inputs": N, "outputs": 15, "examples": int(len(X)), "move_balanced": bal, "move_checked": int(len(yv)),
           "move_fewest": int(moves_seen[moves_seen > 0].min()) if (moves_seen > 0).any() else 0,
           "buttons": QUALITY, "press_lines": THR}, open("/content/policy_info.json", "w"))
print("policy_info.json: put it in Download/MLBB-Research next to policy.tflite - the app then uses only what is reliable:")
print(f"  joystick: {'YES' if bal >= 0.40 and len(yv) >= 300 and moves_seen[moves_seen > 0].min() >= 15 else 'not yet'}"
      f" (needs 40%+ with every move counted equally, 300+ checked moments, 15+ of each move; now {bal:.0%}, {len(yv)}, {moves_seen[moves_seen > 0].min() if (moves_seen > 0).any() else 0})")
for k, q in QUALITY.items():
    print(f"  {k:6s}: {'YES' if q['checked'] >= 20 and q['right'] >= 0.4 else 'not yet'} (needs 20+ presses checked and 40%+ of its presses right; now {q['checked']}, {q['right']:.0%})")
print("policy.tflite:", len(tfl) // 1024, "KB")

# quick check: the .tflite gives the same answer as the trained model
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
it.set_tensor(it.get_input_details()[0]["index"], Xva[:1]); it.invoke()
out = it.get_tensor(it.get_output_details()[0]["index"])[0]
print("check:", np.allclose(out[:9], pm_all[0], atol=1e-4), "| button press lines built in:", dict(zip(BTN, [round(t, 2) for t in THR])))

## 7. Download

In [ ]:
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/policy.tflite"); files.download("/content/policy_info.json")

In [ ]:
# B. Save to Google Drive (most reliable on a phone)
from google.colab import drive; import shutil
drive.mount("/content/drive")
shutil.copy("/content/policy.tflite", "/content/drive/MyDrive/policy.tflite")
shutil.copy("/content/policy_info.json", "/content/drive/MyDrive/policy_info.json")
print("Saved to Google Drive > My Drive: policy.tflite and policy_info.json")

## 8. Put it in the app
1. Put `policy.tflite` AND `policy_info.json` in **Download/MLBB-Research** on the phone; the app loads them by itself.
2. With policy_info.json the app only uses the joystick / the buttons that passed the checks above (without it, all of them).
3. The Player page shows *Tactical network: ready* and, while playing, what it suggests and how often it agrees with the rules.
4. Keep playing and re-train every few matches: it gets better with more of your play.
